In [1]:
import pandas as pd 

In [3]:
df = pd.read_csv('./data/synthetic_learning_data.csv')

In [4]:
df

,student_id,course_id,chapter_order,time_spent,score,completion_status
0,S001,C3,1,53.2,60.8,1
1,S001,C3,2,27.0,100.0,1
2,S001,C3,3,43.6,56.1,1
3,S001,C3,4,31.7,63.8,1
4,S001,C3,5,37.8,72.5,1
...,...,...,...,...,...,...
955,S120,C3,4,54.3,66.9,0
956,S120,C3,5,37.7,77.0,0
957,S120,C3,6,37.9,74.0,0
958,S120,C3,7,36.6,48.1,0


In [5]:
df.columns

Index(['student_id', 'course_id', 'chapter_order', 'time_spent', 'score',
       'completion_status'],
      dtype='object')

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 960 entries, 0 to 959
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   student_id         960 non-null    object 
 1   course_id          960 non-null    object 
 2   chapter_order      960 non-null    int64  
 3   time_spent         960 non-null    float64
 4   score              960 non-null    float64
 5   completion_status  960 non-null    int64  
dtypes: float64(2), int64(2), object(2)
memory usage: 45.1+ KB


In [11]:
df["chapter_order"].unique()


array([1, 2, 3, 4, 5, 6, 7, 8], dtype=int64)

In [13]:
df["course_id"].unique()

array(['C3', 'C2', 'C1'], dtype=object)

In [14]:
df["chapter_order"].unique()

array([1, 2, 3, 4, 5, 6, 7, 8], dtype=int64)

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 960 entries, 0 to 959
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   student_id         960 non-null    object 
 1   course_id          960 non-null    object 
 2   chapter_order      960 non-null    int64  
 3   time_spent         960 non-null    float64
 4   score              960 non-null    float64
 5   completion_status  960 non-null    int64  
dtypes: float64(2), int64(2), object(2)
memory usage: 45.1+ KB


In [18]:
df.drop(columns=["student_id"], inplace=True)

In [19]:
numerical_cols = ["chapter_order","time_spent","score"]
categorical_cols = ["course_id"]

In [20]:
TARGET_COL = "completion_status"

In [21]:
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]


In [22]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

num_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


In [23]:
from sklearn.preprocessing import OneHotEncoder

cat_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])


In [24]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ("num_pipeline", num_pipeline, numerical_cols),
        ("cat_pipeline", cat_pipeline, categorical_cols)
    ]
)


In [25]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)


In [26]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_model(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1_score": f1_score(y_true, y_pred)
    }


In [28]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [29]:
models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "DecisionTree": DecisionTreeClassifier(),
    "RandomForest": RandomForestClassifier()
}


In [30]:
trained_models = {}

for name, model in models.items():

    clf = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    clf.fit(X_train, y_train)

    y_train_pred = clf.predict(X_train)
    y_test_pred = clf.predict(X_test)

    train_metrics = evaluate_model(y_train, y_train_pred)
    test_metrics = evaluate_model(y_test, y_test_pred)

    print(f"\n{name}")
    print("Training Metrics:", train_metrics)
    print("Testing Metrics:", test_metrics)

    # Overfitting check
    if train_metrics["f1_score"] - test_metrics["f1_score"] > 0.10:
        print(f"⚠️ Possible Overfitting Detected in {name}")

    trained_models[name] = {
        "pipeline": clf,
        "test_f1": test_metrics["f1_score"]
    }

    print("=" * 50)



LogisticRegression
Training Metrics: {'accuracy': 0.6502976190476191, 'precision': 0.6511976047904192, 'recall': 0.9954233409610984, 'f1_score': 0.7873303167420815}
Testing Metrics: {'accuracy': 0.6388888888888888, 'precision': 0.6466431095406361, 'recall': 0.9786096256684492, 'f1_score': 0.7787234042553192}

DecisionTree
Training Metrics: {'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1_score': 1.0}
Testing Metrics: {'accuracy': 0.5243055555555556, 'precision': 0.6420454545454546, 'recall': 0.6042780748663101, 'f1_score': 0.6225895316804407}
⚠️ Possible Overfitting Detected in DecisionTree

RandomForest
Training Metrics: {'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1_score': 1.0}
Testing Metrics: {'accuracy': 0.5486111111111112, 'precision': 0.631336405529954, 'recall': 0.732620320855615, 'f1_score': 0.6782178217821783}
⚠️ Possible Overfitting Detected in RandomForest
